# CSC357 Brain-Inspired Artificial Intelligence
## Lab 2 - Neural modelling

This lab will demostrate how to implement the integrate and fire model in Python.

This lab is modified from the Neuromatch Academy tutorial (https://compneuro.neuromatch.io/tutorials/W2D3_BiologicalNeuronModels/student/W2D3_Tutorial1.html). You are welcome to check it if interested.

**Please note that the answers from this lab need to be uploaded as a part of the coursework of CSC357. You can find the questions at the end of this notebook.**

# The Integrate and Fire (IF) model

Let's review the IF model, which is defined by an ordinary differential equation of a neuron's membrane potential $V$:

$$
\tau\frac{dV}{dt} = -(V-V_{rest})+\frac{I(t)}{g}
$$
*   $I(t)$ is the input current to the neuron. For simplicity, we consider I(t) to be a constant in this lab.
*   $\tau$ is the time constant, and $g$ is the condunctance of the synapse, which is also a constant.

The model generate a spike (action potential) when $V$ exceeds a threshold $V>V_{th}$, after which $V$ is reset to the resting potential $V_{rest}$.

Below is an existing implementation of the IF model.

In [ ]:
# First we need to load some libraries for this lab
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Font setting for plots
font = {'family' : 'monospace',
        'weight' : 'normal',
        'size'   : 12}
matplotlib.rc('font', **font)
matplotlib.rc('axes', labelsize=14)

In [ ]:
# Next, we define all the parameters of the IF model and their default values
# Note that we can change those values when calling the function

def default_pars(**kwargs):
  pars = {}

  # default neuron parameters #
  pars['V_th'] = -55.     # spike threshold [mV]
  pars['V_reset'] = -75.  # reset potential [mV]
  pars['tau_m'] = 10.     # membrane time constant [ms]
  pars['g_L'] = 10.       # leak conductance [nS]
  pars['V_init'] = -75.   # initial potential [mV]
  pars['E_L'] = -75.      # leak reversal potential [mV]
  pars['tref'] = 0.       # refractory time (ms)

  # simulation parameters #
  pars['T'] = 400.  # Total duration of simulation [ms]
  pars['dt'] = .1   # Simulation time step [ms]

  # external parameters if any #
  for k in kwargs:
    pars[k] = kwargs[k]

  pars['range_t'] = np.arange(0, pars['T'], pars['dt'])  # Vector of discretized time points [ms]

  return pars

In [ ]:
# This is the core function of the IF model
def run_LIF(pars, Iinj, stop=False, Idur=1000):
  """
  Simulate the LIF dynamics with external input current

  Args:
    pars       : parameter dictionary
    Iinj       : input current [pA]. The injected current here can be a value
                 or an array
    stop       : boolean. If True, use a current pulse
    Idur       : the duration of Iinj in ms. Only works if stop=True

  Returns:
    rec_v      : membrane potential
    rec_sp     : spike times
  """

  # Set parameters
  V_th, V_reset = pars['V_th'], pars['V_reset']
  tau_m, g_L = pars['tau_m'], pars['g_L']
  V_init, E_L = pars['V_init'], pars['E_L']
  dt, range_t = pars['dt'], pars['range_t']
  Lt = range_t.size
  tref = pars['tref']

  # Initialize voltage
  v = np.zeros(Lt)
  v[0] = V_init

  # Set current time course
  Iinj = Iinj * np.ones(Lt)

  # If current pulse, set beginning and end to 0
  # We use a constant current pulse with a duration of 1000 ms
  if stop:
    Iinj[:int(len(Iinj) / 2) - int(Idur/2/dt)] = 0
    Iinj[int(len(Iinj) / 2) + int(Idur/2/dt):] = 0

  # Loop over time
  rec_spikes = []   # record spike times
  tr = 0.           # the count for refractory duration

  # The key implemention of the IF model
  for it in range(Lt - 1):

    if tr > 0:         # check if in refractory period
      v[it] = V_reset  # set voltage to reset
      tr = tr - 1      # reduce running counter of refractory period

    elif v[it] >= V_th:       # if voltage over threshold
      rec_spikes.append(it)   # record spike event
      v[it] = V_reset         # reset voltage
      tr = tref / dt          # set refractory time

    # Calculate the increment of the membrane potential using the Euler method
    dv = (-(v[it] - E_L) + Iinj[it] / g_L) * (dt / tau_m)

    # Update the membrane potential
    v[it + 1] = v[it] + dv

  # Get spike times in ms
  rec_spikes = np.array(rec_spikes) * dt

  return v, rec_spikes, Iinj

In [ ]:
# This is an auxiliary function for plotting
def plot_volt_trace(pars, v, sp):
  """
  Plot trajetory of membrane potential for a single neuron

  Expects:
  pars   : parameter dictionary
  v      : volt trajetory
  sp     : spike train

  Returns:
  figure of the membrane potential trajetory for a single neuron
  """

  V_th = pars['V_th']
  dt, range_t = pars['dt'], pars['range_t']
  if sp.size:
    sp_num = (sp / dt).astype(int) - 1
    v[sp_num] += 20  # draw nicer spikes

  plt.plot(pars['range_t'], v, 'b')
  plt.axhline(V_th, 0, 1, color='k', ls='--')
  plt.xlabel('Time (ms)')
  plt.ylabel('V (mV)')
  plt.legend(['Membrane\npotential', r'Threshold V$_{\mathrm{th}}$'],
             loc=[1.05, 0.75])
  plt.ylim([-80, -40])
  plt.show()

def plot_I_trace(pars, I):
  """
  Plot trajetory of the input current

  Expects:
  pars   : parameter dictionary
  I      : input current

  Returns:
  figure of the input current
  """
  dt = pars['dt']
  plt.plot(pars['range_t'], I, 'b')
  plt.xlabel('Time (ms)')
  plt.ylabel('I (pA)')
  plt.legend(['Input\ncurrent'],loc=[1.05, 0.75])
  plt.show()


### All set now. Let's run a simple simulation.

**<span style="color:blue">TODO: Run the simulation below and observe the results</span>**

In [ ]:
# Initilize parameters, setting total simulation time T=1400 ms and the refractory period is 0 ms.
pars = default_pars(T=1400, tref=0)

# Simulate LIF model, using constant input current Iinj=230 with a duration of 1000 ms
v, sp, Iinj = run_LIF(pars, Iinj=230, stop=True, Idur=1000)

# Visualize the membrane potential and the input current, also text output on the total number of spikes
plot_volt_trace(pars, v, sp)
plot_I_trace(pars, Iinj)
print('Total number of spikes within 1000 ms: '+str(sp.size))

## **Coursework TODO**

You should observe that the specified setting above (input current $I_{inj} = 230$) generates spike trains. Using the provided scripts as a guide, please answer the following two questions, and include your answers in your coursework submission.


---


**[Q1]**. Maintain all other parameters unchanged and re-run the simulation with lower values of $I_{inj}$. Can you pinpoint the critical $I_{inj}$
value that fails to produce any spike?

**In your coursework, include a copy of the membrane potential trace figure generated from `plot_volt_trace()`, simulated under the critical $I_{inj}$
value**.


---


**[Q2]**. Maintain all other parameters at default values(including $𝐼_{𝑖𝑛𝑗}=230$), and run the model simulation with different refractory periods. You can do so by re-initializing the parameter, e.g., `pars = default_pars(T=1400, tref=5)`.

**In your coursework, give a statement on how increasing the refractory period impacts the number of spikes generated per unit of time.**



In [ ]:
# You can also explore the effect of other parameters on the model behaviour.
